# 03 - Unity Catalog governance

* **Column masks** on PII (`device_id`, `ip`, `phone`): members of the privileged group (`pii_admin_group`, default `admins`) see clear values; everyone else sees masked values.
  * `device_id` is masked to a **deterministic token** (`dev-` + sha256 prefix) so non-admin consumers (analysts, feature jobs running as a non-admin principal) can still count distinct devices / join on device without seeing the raw identifier.
  * `ip` keeps the first two octets; `phone` keeps the last 4 digits.
* Masks on pipeline-managed silver tables are declared **in the pipeline schema** (`MASK` clause) because SDP owns those tables; masks on the regular `customers` table are applied here with `ALTER TABLE`.
* Bronze keeps the raw payload (`data` struct) for replay/debugging and is treated as an **engineering-only** layer: no grants to analyst groups.
* Table comments + tags (`domain`, `layer`, `contains_pii`, `data_origin=synthetic`) and column tags (`pii=...`).

Run order: cells 2-3 before the pipeline update (functions must exist), the rest after.

In [0]:
import re

dbutils.widgets.text("catalog", "mlo_serverless_stable_catalog")
dbutils.widgets.text("schema", "fraud_rt")
dbutils.widgets.text("pii_admin_group", "admins")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
PII_GROUP = dbutils.widgets.get("pii_admin_group")
for v in (CATALOG, SCHEMA, PII_GROUP):  # these values are interpolated into DDL -> allow identifiers only
    assert re.fullmatch(r"[A-Za-z0-9_\-]+", v), f"invalid identifier: {v}"
FQ = f"`{CATALOG}`.`{SCHEMA}`"

# This shared workspace has ~1000 account-level governed tag policies (e.g. 'pii', 'pii_type', 'domain'-like keys with
# restricted values). To avoid collisions, custom tags use a project prefix and PII columns also get the built-in
# data-classification keys (class.phone_number / class.ip_address) so they line up with UC data classification / ABAC.
P = "fraud_rt_"
COLUMN_CLASS_TAG = {"phone": "class.phone_number", "ip": "class.ip_address", "home_ip": "class.ip_address"}

def pii_column_tags(col):
    kind = {"home_ip": "ip", "primary_device_id": "device_id"}.get(col, col)
    tags = [f"'{P}pii_type' = '{kind}'"]
    if col in COLUMN_CLASS_TAG:
        tags.append(f"'{COLUMN_CLASS_TAG[col]}' = ''")  # key-only governed tag
    return ", ".join(tags)

print(FQ, PII_GROUP, pii_column_tags("phone"))

In [0]:
PRIV = f"(is_member('{PII_GROUP}') OR is_account_group_member('{PII_GROUP}'))"

masks = {
    "mask_device_id": ("Masks device_id for non-privileged users with a deterministic token (joinable, countable, not reversible).",
                       "concat('dev-', substr(sha2(v, 256), 1, 12))"),
    "mask_ip": ("Masks IPv4 for non-privileged users: keeps first two octets.",
                "concat(substring_index(v, '.', 2), '.***.***')"),
    "mask_phone": ("Masks phone numbers for non-privileged users: keeps last 4 digits.",
                   "concat('***-***-', right(v, 4))"),
}
for fn, (comment, masked_expr) in masks.items():
    spark.sql(f"""
        CREATE OR REPLACE FUNCTION {FQ}.{fn}(v STRING)
        RETURNS STRING
        COMMENT '{comment} Privileged group: {PII_GROUP}.'
        RETURN CASE WHEN v IS NULL THEN NULL WHEN {PRIV} THEN v ELSE {masked_expr} END
    """)
display(spark.sql(f"SELECT routine_name, comment FROM `{CATALOG}`.information_schema.routines WHERE routine_schema = '{SCHEMA}' ORDER BY 1"))

In [0]:
T = f"{FQ}.customers"
spark.sql(f"COMMENT ON TABLE {T} IS 'Synthetic customer dimension for the remittance fraud prototype (5K customers). Contains masked PII (phone, primary_device_id, home_ip). No fraud label by design.'")
col_comments = {
    "customer_id": "Synthetic customer key (C000001..). Join key to all event tables.",
    "home_corridor": "Corridor the customer usually sends to, e.g. US-MX.",
    "preferred_payout": "Usual payout method: bank_deposit, cash_pickup, mobile_wallet, home_delivery.",
    "phone": "PII. Fictional +1-555 phone number. Masked for non-privileged users.",
    "primary_device_id": "PII. Primary device identifier of the customer. Masked (deterministic token) for non-privileged users.",
    "home_ip": "PII. Usual IP of the customer (CGNAT range, synthetic). Masked for non-privileged users.",
    "signup_date": "Account signup date.",
}
for c, txt in col_comments.items():
    spark.sql(f"ALTER TABLE {T} ALTER COLUMN {c} COMMENT '{txt}'")
for c, fn in {"phone": "mask_phone", "primary_device_id": "mask_device_id", "home_ip": "mask_ip"}.items():
    spark.sql(f"ALTER TABLE {T} ALTER COLUMN {c} SET MASK {FQ}.{fn}")
    spark.sql(f"ALTER TABLE {T} ALTER COLUMN {c} SET TAGS ({pii_column_tags(c)})")
spark.sql(f"ALTER TABLE {T} SET TAGS ('{P}domain' = 'fraud', '{P}layer' = 'dimension', '{P}contains_pii' = 'true', '{P}data_origin' = 'synthetic')")
# cleanup: an earlier attempt set the governed 'pii_type' key on phone before switching to the prefixed key
spark.sql(f"ALTER TABLE {T} ALTER COLUMN phone UNSET TAGS ('pii_type')")
print("customers governed")

In [0]:
PII_COLS = {"login": ["device_id", "ip"], "otp": ["device_id", "phone"], "screen_view": ["device_id"], "transfers": ["device_id", "ip"]}
for s, cols in PII_COLS.items():
    spark.sql(f"ALTER TABLE {FQ}.bronze_{s} SET TAGS ('{P}domain' = 'fraud', '{P}layer' = 'bronze', '{P}contains_pii' = 'true', '{P}access' = 'engineering_only', '{P}data_origin' = 'synthetic', '{P}source' = 'kinesis_shaped_json')")
    spark.sql(f"ALTER TABLE {FQ}.silver_{s} SET TAGS ('{P}domain' = 'fraud', '{P}layer' = 'silver', '{P}contains_pii' = 'true', '{P}data_origin' = 'synthetic', '{P}cdf' = 'enabled')")
    for c in cols:
        spark.sql(f"ALTER TABLE {FQ}.silver_{s} ALTER COLUMN {c} SET TAGS ({pii_column_tags(c)})")
spark.sql(f"ALTER TABLE {FQ}.silver_transfers ALTER COLUMN is_fraud SET TAGS ('{P}ml_role' = 'label')")
print("pipeline tables tagged")

In [0]:
display(spark.sql(f"""
    SELECT table_name, column_name, mask_name FROM `{CATALOG}`.information_schema.column_masks
    WHERE table_schema = '{SCHEMA}' ORDER BY 1, 2"""))
display(spark.sql(f"""
    SELECT table_name, concat_ws(', ', collect_list(concat(tag_name, '=', tag_value))) AS tags
    FROM `{CATALOG}`.information_schema.table_tags WHERE schema_name = '{SCHEMA}' GROUP BY 1 ORDER BY 1"""))
display(spark.sql(f"""
    SELECT table_name, column_name, tag_name, tag_value FROM `{CATALOG}`.information_schema.column_tags
    WHERE schema_name = '{SCHEMA}' ORDER BY 1, 2"""))
display(spark.sql(f"""
    SELECT table_name, comment FROM `{CATALOG}`.information_schema.tables
    WHERE table_schema = '{SCHEMA}' AND table_name NOT LIKE '\\_\\_materialization%' ORDER BY 1"""))

In [0]:
# Current user is in the privileged group, so table reads return clear values.
# The "non_privileged_view" columns evaluate the exact ELSE branch of each mask function to show what other users get.
display(spark.sql(f"""
    SELECT customer_id, event_ts, status,
           phone AS phone_as_seen_by_me, concat('***-***-', right(phone, 4)) AS phone_non_privileged_view,
           device_id AS device_as_seen_by_me, concat('dev-', substr(sha2(device_id, 256), 1, 12)) AS device_non_privileged_view,
           (is_member('{PII_GROUP}') OR is_account_group_member('{PII_GROUP}')) AS i_am_privileged
    FROM {FQ}.silver_otp WHERE status = 'failed' LIMIT 5"""))